In [1]:
import pandas as pd
import numpy as np
import tqdm as notebook_tqdm
import hopsworks
import os

d:\Virtual_Environments\10pearls_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
df = pd.read_csv(r"..\Data\sample_data.csv")
df.head()

,time,temperature_2m,relative_humidity_2m,wind_speed_10m,surface_pressure,cloud_cover,precipitation,us_aqi,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,month,hour
0,2023-09-05 00:00:00,27.1,75,8.8,985.8,0,0.0,88,44.1,68.2,970.0,49.3,6.6,17.0,28.0,9,0
1,2023-09-05 01:00:00,27.5,70,6.8,985.7,0,0.0,87,35.9,61.5,763.0,37.5,5.0,25.0,50.0,9,1
2,2023-09-05 02:00:00,27.2,70,7.7,985.8,0,0.0,86,32.4,70.5,582.0,27.7,3.8,31.0,69.0,9,2
3,2023-09-05 03:00:00,27.2,71,9.1,985.9,0,0.0,87,31.7,88.7,470.0,22.7,3.2,31.0,85.0,9,3
4,2023-09-05 04:00:00,27.2,71,10.3,986.0,0,0.0,87,29.8,92.4,384.0,19.7,3.0,30.0,99.0,9,4


In [18]:
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values("time").reset_index(drop=True)

df["hour"] = df["time"].dt.hour
df["day"] = df["time"].dt.day
df["month"] = df["time"].dt.month
df["day_of_year"] = df["time"].dt.dayofyear
df["day_of_week"] = df["time"].dt.dayofweek 

# Cyclical encoding — so Dec and Jan (both winter) end up numerically close
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

print(df[["time", "hour", "month", "month_sin", "month_cos", "hour_sin", "hour_cos"]].head())

                 time  hour  month  month_sin     month_cos  hour_sin  \
0 2023-09-05 00:00:00     0      9       -1.0 -1.836970e-16  0.000000   
1 2023-09-05 01:00:00     1      9       -1.0 -1.836970e-16  0.258819   
2 2023-09-05 02:00:00     2      9       -1.0 -1.836970e-16  0.500000   
3 2023-09-05 03:00:00     3      9       -1.0 -1.836970e-16  0.707107   
4 2023-09-05 04:00:00     4      9       -1.0 -1.836970e-16  0.866025   

   hour_cos  
0  1.000000  
1  0.965926  
2  0.866025  
3  0.707107  
4  0.500000  


In [19]:
expected_range = pd.date_range(df["time"].min(), df["time"].max(), freq="h")
missing_hours = expected_range.difference(df["time"])

print(f"Expected hourly rows: {len(expected_range)}")
print(f"Actual rows: {len(df)}")
print(f"Missing hours: {len(missing_hours)}")

if len(missing_hours) > 0:
    print(missing_hours[:20])
    raise ValueError(
        f"{len(missing_hours)} missing hour(s) in the timeline -- lag/rolling/"
        "target features would be misaligned across each gap. Reindex and "
        "interpolate (or re-fetch the missing range from Open-Meteo) before "
        "proceeding with feature engineering."
    )

Expected hourly rows: 26304
Actual rows: 26304
Missing hours: 0


In [20]:
lag_hours = [1, 3, 6, 12, 24]

for lag in lag_hours:
    df[f"aqi_lag_{lag}h"] = df["us_aqi"].shift(lag)
    df[f"wind_speed_lag_{lag}h"] = df["wind_speed_10m"].shift(lag)
    df[f"pressure_lag_{lag}h"] = df["surface_pressure"].shift(lag)

print(df[["time", "us_aqi", "aqi_lag_1h", "aqi_lag_6h", "aqi_lag_24h"]].head(30))

                  time  us_aqi  aqi_lag_1h  aqi_lag_6h  aqi_lag_24h
0  2023-09-05 00:00:00      88         NaN         NaN          NaN
1  2023-09-05 01:00:00      87        88.0         NaN          NaN
2  2023-09-05 02:00:00      86        87.0         NaN          NaN
3  2023-09-05 03:00:00      87        86.0         NaN          NaN
4  2023-09-05 04:00:00      87        87.0         NaN          NaN
5  2023-09-05 05:00:00      88        87.0         NaN          NaN
6  2023-09-05 06:00:00      89        88.0        88.0          NaN
7  2023-09-05 07:00:00      89        89.0        87.0          NaN
8  2023-09-05 08:00:00      89        89.0        86.0          NaN
9  2023-09-05 09:00:00      89        89.0        87.0          NaN
10 2023-09-05 10:00:00      90        89.0        87.0          NaN
11 2023-09-05 11:00:00      90        90.0        88.0          NaN
12 2023-09-05 12:00:00      91        90.0        89.0          NaN
13 2023-09-05 13:00:00      91        91.0      

In [21]:
# Rolling averages — smoothed recent trend
df["aqi_roll_mean_6h"] = df["us_aqi"].rolling(window=6).mean()
df["aqi_roll_mean_24h"] = df["us_aqi"].rolling(window=24).mean()
df["aqi_roll_std_24h"] = df["us_aqi"].rolling(window=24).std()

# AQI change rate — is it rising or falling, and how fast
df["aqi_change_1h"] = df["us_aqi"].diff(1)
df["aqi_change_6h"] = df["us_aqi"].diff(6)

print(df[["time", "us_aqi", "aqi_roll_mean_6h", "aqi_roll_mean_24h", "aqi_change_1h"]].head(30))

                  time  us_aqi  aqi_roll_mean_6h  aqi_roll_mean_24h  \
0  2023-09-05 00:00:00      88               NaN                NaN   
1  2023-09-05 01:00:00      87               NaN                NaN   
2  2023-09-05 02:00:00      86               NaN                NaN   
3  2023-09-05 03:00:00      87               NaN                NaN   
4  2023-09-05 04:00:00      87               NaN                NaN   
5  2023-09-05 05:00:00      88         87.166667                NaN   
6  2023-09-05 06:00:00      89         87.333333                NaN   
7  2023-09-05 07:00:00      89         87.666667                NaN   
8  2023-09-05 08:00:00      89         88.166667                NaN   
9  2023-09-05 09:00:00      89         88.500000                NaN   
10 2023-09-05 10:00:00      90         89.000000                NaN   
11 2023-09-05 11:00:00      90         89.333333                NaN   
12 2023-09-05 12:00:00      91         89.666667                NaN   
13 202

In [22]:
print("Shape before cleanup:", df.shape)
print("Rows with any NaN:", df.isnull().any(axis=1).sum())

df = df.dropna().reset_index(drop=True)

print("Shape after cleanup:", df.shape)

Shape before cleanup: (26304, 44)
Rows with any NaN: 24
Shape after cleanup: (26280, 44)


In [23]:
df = df.sort_values('time').reset_index(drop=True)

In [24]:
# Multi-horizon targets for 3-day forecast (24h, 48h, 72h ahead)
df['target_aqi_24h'] = df['us_aqi'].shift(-24)
df['target_aqi_48h'] = df['us_aqi'].shift(-48)
df['target_aqi_72h'] = df['us_aqi'].shift(-72)

print(df[['time', 'us_aqi', 'target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']].tail(75))

                     time  us_aqi  target_aqi_24h  target_aqi_48h  \
26205 2026-09-01 21:00:00     101           103.0            88.0   
26206 2026-09-01 22:00:00     101           102.0            89.0   
26207 2026-09-01 23:00:00     102           101.0            89.0   
26208 2026-09-02 00:00:00     102            99.0            89.0   
26209 2026-09-02 01:00:00     102            98.0            89.0   
...                   ...     ...             ...             ...   
26275 2026-09-04 19:00:00      84             NaN             NaN   
26276 2026-09-04 20:00:00      84             NaN             NaN   
26277 2026-09-04 21:00:00      84             NaN             NaN   
26278 2026-09-04 22:00:00      84             NaN             NaN   
26279 2026-09-04 23:00:00      84             NaN             NaN   

       target_aqi_72h  
26205            84.0  
26206            84.0  
26207            84.0  
26208             NaN  
26209             NaN  
...               ...  
262

In [9]:
print(df.shape)
print(df['time'].min(), df['time'].max())

(26280, 47)
2023-09-06 00:00:00 2026-09-04 23:00:00


In [10]:
os.makedirs(r"D:\tmp", exist_ok=True)

project = hopsworks.login(
    api_key_value=os.getenv("AQI_Predictor_KEY"),
    cert_folder="./hopsworks-certs"
)
fs = project.get_feature_store()

fg_v2 = fs.get_or_create_feature_group(
    name="aqi_features_multan",
    version=2,
    primary_key=["time"],
    event_time="time",
    time_travel_format="HUDI", 
    description="AQI features for Multan with 24h/48h/72h multi-horizon targets",
    online_enabled=False,  
    stream=False,
)

2026-09-05 07:46:42,486 INFO: Initializing external client
2026-09-05 07:46:42,489 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-05 07:46:51,209 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41139


In [ ]:
os.makedirs(r"D:\tmp", exist_ok=True)

result = fg_v2.insert(df, write_options={"wait_for_job": True})
job = result[0] if isinstance(result, tuple) else result

final_state = job.get_state() if job is not None else None
print("Materialization job final state:", final_state)

if final_state not in (None, "SUCCEEDED", "FINISHED"):
    raise RuntimeError(
        f"Feature Group insert job did not succeed (state={final_state}). "
        "Check the Hopsworks job logs before trusting downstream reads."
    )

In [12]:
fg_v2 = fs.get_feature_group(name="aqi_features_multan", version=2)
df = fg_v2.read()
df = df.sort_values('time').reset_index(drop=True)
print(df.shape)
print(df[['time', 'us_aqi', 'target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']].tail(100))

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (14.75s) 
(26280, 47)
                           time  us_aqi  target_aqi_24h  target_aqi_48h  \
26180 2026-08-31 20:00:00+00:00     100           101.0           103.0   
26181 2026-08-31 21:00:00+00:00     100           101.0           103.0   
26182 2026-08-31 22:00:00+00:00     100           101.0           102.0   
26183 2026-08-31 23:00:00+00:00     100           102.0           101.0   
26184 2026-09-01 00:00:00+00:00      99           102.0            99.0   
...                         ...     ...             ...             ...   
26275 2026-09-04 19:00:00+00:00      84             NaN             NaN   
26276 2026-09-04 20:00:00+00:00      84             NaN             NaN   
26277 2026-09-04 21:00:00+00:00      84             NaN             NaN   
26278 2026-09-04 22:00:00+00:00      84             NaN             NaN   
26279 2026-09-04 23:00:00+00:00      84             NaN             NaN   



In [13]:
lag_roll_cols = [c for c in df.columns if "lag" in c or "roll" in c]

print("Total rows:", df.shape)
print("Min date:", df["time"].min())
print("Max date:", df["time"].max())
print("Rows with NaN in lag/roll cols:", df[lag_roll_cols].isnull().any(axis=1).sum())

Total rows: (26280, 47)
Min date: 2023-09-06 00:00:00+00:00
Max date: 2026-09-04 23:00:00+00:00
Rows with NaN in lag/roll cols: 0
